# Spark Connect - Introduction

Spark Connect는 Spark 4.x의 클라이언트-서버 아키텍처입니다.

- **기존**: Driver가 Spark 클러스터 안에서 실행 (tight coupling)
- **Spark Connect**: Client(Jupyter)가 gRPC로 원격 Spark 서버에 접속 (decoupled)

```
Jupyter (Client) ──gRPC──> Spark Connect Server ──> Executors
```

## 1. Spark Session 생성

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .remote("sc://spark:15002") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 2. DataFrame 기본 연산

In [ ]:
# 간단한 DataFrame 생성
df = spark.range(1000)
df.show(5)

In [ ]:
from pyspark.sql import functions as F

# 변환 연산
result = df.withColumn("squared", F.col("id") ** 2) \
           .withColumn("is_even", F.col("id") % 2 == 0) \
           .filter(F.col("id") < 20)

result.show()

## 3. Python 데이터 → Spark DataFrame

In [ ]:
import pandas as pd

# Pandas DataFrame → Spark DataFrame
pdf = pd.DataFrame({
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [28, 35, 42, 31],
    "dept": ["Engineering", "Marketing", "Engineering", "Marketing"]
})

sdf = spark.createDataFrame(pdf)
sdf.show()
sdf.printSchema()

## 4. Spark SQL

In [ ]:
# Temp View 등록 후 SQL 실행
sdf.createOrReplaceTempView("employees")

spark.sql("""
    SELECT dept, 
           COUNT(*) as count, 
           AVG(age) as avg_age
    FROM employees
    GROUP BY dept
""").show()

## 5. CSV 파일 읽기 (data/ 볼륨)

In [ ]:
# data/ 폴더에 CSV가 있으면 읽기
# Spark Connect 서버에서 마운트된 경로: /opt/spark/work/data/
# 
# df_csv = spark.read.csv("/opt/spark/work/data/sample.csv", header=True, inferSchema=True)
# df_csv.show()

In [ ]:
spark.stop()